# Models for Future value prediction and to find if its a good investment

### Library Imports

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import (LinearRegression,Ridge,Lasso,LogisticRegression)
from sklearn.ensemble import (RandomForestRegressor,GradientBoostingRegressor,RandomForestClassifier,GradientBoostingClassifier)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor,XGBClassifier
from lightgbm import LGBMRegressor,LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    confusion_matrix,
    precision_score,
    f1_score,
    recall_score,
    roc_curve,
    roc_auc_score,
    accuracy_score
)
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import mlflow.xgboost
import mlflow.lightgbm
from mlflow.tracking import MlflowClient

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
reg_experiment_name = "real_estate_regression"
clf_experiment_name = "real_estate_regression"

### Helper Functions

In [ ]:

def log_model_by_type(model, name, signature):
    # xgboost's and lightgbm's Booster/estimator classes (and OrderedDict used
    # internally by lightgbm) are flagged as untrusted by skops (mlflow's sklearn
    # serializer); we trust them since we trained them ourselves.
    trusted_types = [
        "xgboost.core.Booster", "xgboost.sklearn.XGBRegressor", "xgboost.sklearn.XGBClassifier",
        "lightgbm.basic.Booster", "lightgbm.sklearn.LGBMRegressor", "lightgbm.sklearn.LGBMClassifier",
        "collections.OrderedDict",
    ]
    mlflow.sklearn.log_model(model, name="model", signature=signature, skops_trusted_types=trusted_types)


def regression_metrics(y_true, preds):
    return {
        "rmse": np.sqrt(mean_squared_error(y_true, preds)),
        "mae": mean_absolute_error(y_true, preds),
        "r2": r2_score(y_true, preds),
    }


def classification_metrics(y_true, preds, proba):
    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds),
        "recall": recall_score(y_true, preds),
        "f1": f1_score(y_true, preds),
        "roc_auc": roc_auc_score(y_true, proba),
    }

def set_mlflow_experiment(experiment_name):
    mlflow.set_experiment(experiment_name)


In [ ]:
from sklearn.model_selection import RandomizedSearchCV


def tune_lgbm_random_search(estimator, param_dist, X_train, y_train, scoring,
                             n_iter=20, cv=3, random_state=42, n_jobs=-1, verbose=1):
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", estimator),
    ])

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=cv,
        scoring=scoring,
        random_state=random_state,
        n_jobs=n_jobs,
        verbose=verbose,
    )

    search.fit(X_train, y_train)

    print("Best params:", search.best_params_)
    print(f"Best CV {scoring}:", round(search.best_score_, 4))

    return search

### Data Load and Train Test Split

In [24]:
df = pd.read_csv(r'Datasets/india_housing_prices_with_target_columns_encoded.csv')
df.head()

,State,Locality,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Floor_No,Total_Floors,Age_of_Property,...,Facing_East,Facing_North,Facing_South,Facing_West,Furnished_Status_Furnished,Furnished_Status_Semi_Furnished,Furnished_Status_Unfurnished,Owner_Type_Broker,Owner_Type_Builder,Owner_Type_Owner
0,Tamil Nadu,Locality_84,1,4740,489.76,0.10,1990,22,1,35,...,0,0,0,1,1,0,0,0,0,1
1,Maharashtra,Locality_490,3,2364,195.52,0.08,2008,21,20,17,...,0,1,0,0,0,0,1,0,1,0
2,Punjab,Locality_167,2,3642,183.79,0.05,1997,19,27,28,...,0,0,1,0,0,1,0,1,0,0
3,Rajasthan,Locality_393,2,2741,300.29,0.11,1991,21,26,34,...,0,1,0,0,1,0,0,0,1,0
4,Rajasthan,Locality_466,4,4823,182.90,0.04,2002,3,2,23,...,1,0,0,0,0,1,0,0,1,0


### Regression Model

In [25]:
x_reg_cols = ['BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Floor_No', 'Total_Floors',
    'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals',
    'Public_Transport_Accessibility', 'Parking_Space', 'Security',
    'Availability_Status', 'Amenities_Count',
    'City_Ahmedabad', 'City_Amritsar', 'City_Bangalore', 'City_Bhopal',
    'City_Bhubaneswar', 'City_Bilaspur', 'City_Chennai', 'City_Coimbatore',
    'City_Cuttack', 'City_Dehradun', 'City_Durgapur', 'City_Dwarka',
    'City_Faridabad', 'City_Gaya', 'City_Gurgaon', 'City_Guwahati',
    'City_Haridwar', 'City_Hyderabad', 'City_Indore', 'City_Jaipur',
    'City_Jamshedpur', 'City_Jodhpur', 'City_Kochi', 'City_Kolkata',
    'City_Lucknow', 'City_Ludhiana', 'City_Mangalore', 'City_Mumbai',
    'City_Mysore', 'City_Nagpur', 'City_New_Delhi', 'City_Noida',
    'City_Patna', 'City_Pune', 'City_Raipur', 'City_Ranchi', 'City_Silchar',
    'City_Surat', 'City_Trivandrum', 'City_Vijayawada',
    'City_Vishakhapatnam', 'City_Warangal', 'Property_Type_Apartment',
    'Property_Type_Independent_House', 'Property_Type_Villa', 'Facing_East',
    'Facing_North', 'Facing_South', 'Facing_West',
    'Furnished_Status_Furnished', 'Furnished_Status_Semi_Furnished',
    'Furnished_Status_Unfurnished', 'Owner_Type_Broker',
    'Owner_Type_Builder', 'Owner_Type_Owner']
X_reg = df[x_reg_cols]
y_reg = df['Future_Price_5Y']

In [26]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

In [27]:
models = {
    "LinearRegression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "Ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
    "Lasso": Pipeline([("scaler", StandardScaler()), ("model", Lasso())]),
    "RandomForest": Pipeline([("scaler", StandardScaler()), ("model", RandomForestRegressor(random_state=42))]),
    "GradientBoosting": Pipeline([("scaler", StandardScaler()), ("model", GradientBoostingRegressor(random_state=42))]),
    "XGBoost": Pipeline([("scaler", StandardScaler()), ("model", XGBRegressor(random_state=42))]),
    "LightGBM": Pipeline([("scaler", StandardScaler()), ("model", LGBMRegressor(random_state=42))]),
}

In [28]:

mlflow.set_experiment("real_estate_regression")

<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1787666057911, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1787666057911, lifecycle_stage='active', name='real_estate_regression', tags={}, trace_location=None, workspace='default'>

In [29]:
def train_and_log_regressor(models):
    for name, model in models.items():
        with mlflow.start_run(run_name=name):
            model.fit(X_train_reg, y_train_reg)
            preds = model.predict(X_test_reg)

            metrics = regression_metrics(y_test_reg, preds)

            mlflow.log_param("model_type", name)
            mlflow.log_metrics(metrics)

            signature = infer_signature(X_test_reg, preds)
            log_model_by_type(model, name, signature)

            print(f"{name}: RMSE={metrics['rmse']:.2f}  MAE={metrics['mae']:.2f}  R2={metrics['r2']:.3f}")

In [ ]:
def train_and_log_tuned_regressor(search, name, X_test, y_test):
    best_model = search.best_estimator_

    preds = best_model.predict(X_test)
    metrics = regression_metrics(y_test, preds)

    print(f"Tuned {name} (test): RMSE={metrics['rmse']:.2f}  MAE={metrics['mae']:.2f}  R2={metrics['r2']:.3f}")

    with mlflow.start_run(run_name=f"{name}_tuned") as run:
        mlflow.log_params(search.best_params_)
        mlflow.log_metrics(metrics)
        signature = infer_signature(X_test, preds)
        log_model_by_type(best_model, name, signature)
        run_id = run.info.run_id

    return best_model, metrics, run_id

In [30]:
train_and_log_regressor(models=models)

/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


LinearRegression: RMSE=39.05  MAE=29.32  R2=0.960
🏃 View run LinearRegression at: http://127.0.0.1:5000/#/experiments/4/runs/dedd1c0e3d5d404c8d5757c6bf026680
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Ridge: RMSE=39.05  MAE=29.32  R2=0.960
🏃 View run Ridge at: http://127.0.0.1:5000/#/experiments/4/runs/c69d4b07829344c5bcf2323b1947620c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


...

GradientBoosting: RMSE=37.87  MAE=26.61  R2=0.963
🏃 View run GradientBoosting at: http://127.0.0.1:5000/#/experiments/4/runs/0f4aca13c1174e99b6264815f22f35de
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


XGBoost: RMSE=33.82  MAE=23.47  R2=0.970
🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/4/runs/b3bfb7333b2b440a839a7b13d929befc
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011009 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 820
[LightGBM] [Info] Number of data points in the train set: 200000, number of used features: 68
[LightGBM] [Info] Start training from score 339.333309


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


LightGBM: RMSE=33.45  MAE=23.28  R2=0.971
🏃 View run LightGBM at: http://127.0.0.1:5000/#/experiments/4/runs/dfaec361ff4d482cb945ae43cf9a15e4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [ ]:
param_dist = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [-1, 5, 10, 20],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "model__num_leaves": [31, 50, 70, 100],
    "model__subsample": [0.7, 0.8, 1.0],
}

search = tune_lgbm_random_search(
    LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    param_dist,
    X_train_reg, y_train_reg,
    scoring="r2",
)

In [ ]:
best_lgbm, metrics, run_id = train_and_log_tuned_regressor(
    search, "LightGBM", X_test_reg, y_test_reg
)
print(run_id)

In [33]:
mlflow.register_model(
    f"runs:/{run_id}/model",
    "real_estate_regression_model"
)

Successfully registered model 'real_estate_regression_model'.
2026/08/27 14:32:04 WARNING mlflow.tracking._model_registry.fluent: Run with id 1cc65c2bbd294b6b8bd70f89cceb6ad2 has no artifacts at artifact path 'model', registering model based on models:/m-a7c517e295ce42eea10026be1b237a88 instead
2026/08/27 14:32:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: real_estate_regression_model, version 1
Created version '1' of model 'real_estate_regression_model'.


<ModelVersion: aliases=[], creation_timestamp=1787821324863, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1787821324863, metrics=None, model_id=None, name='real_estate_regression_model', params=None, run_id='1cc65c2bbd294b6b8bd70f89cceb6ad2', run_link='', source='models:/m-a7c517e295ce42eea10026be1b237a88', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

### Classification Model

In [34]:
x_clf_cols = ['Size_in_SqFt', 'Price_in_Lakhs', 'Floor_No', 'Total_Floors',
    'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals',
    'Public_Transport_Accessibility', 'Security',
    'City_Ahmedabad', 'City_Amritsar', 'City_Bangalore', 'City_Bhopal',
    'City_Bhubaneswar', 'City_Bilaspur', 'City_Chennai', 'City_Coimbatore',
    'City_Cuttack', 'City_Dehradun', 'City_Durgapur', 'City_Dwarka',
    'City_Faridabad', 'City_Gaya', 'City_Gurgaon', 'City_Guwahati',
    'City_Haridwar', 'City_Hyderabad', 'City_Indore', 'City_Jaipur',
    'City_Jamshedpur', 'City_Jodhpur', 'City_Kochi', 'City_Kolkata',
    'City_Lucknow', 'City_Ludhiana', 'City_Mangalore', 'City_Mumbai',
    'City_Mysore', 'City_Nagpur', 'City_New_Delhi', 'City_Noida',
    'City_Patna', 'City_Pune', 'City_Raipur', 'City_Ranchi', 'City_Silchar',
    'City_Surat', 'City_Trivandrum', 'City_Vijayawada',
    'City_Vishakhapatnam', 'City_Warangal', 'Property_Type_Apartment',
    'Property_Type_Independent_House', 'Property_Type_Villa', 'Facing_East',
    'Facing_North', 'Facing_South', 'Facing_West',
    'Furnished_Status_Furnished', 'Furnished_Status_Semi_Furnished',
    'Furnished_Status_Unfurnished', 'Owner_Type_Broker',
    'Owner_Type_Builder', 'Owner_Type_Owner']
X_clf = df[x_clf_cols]
y_clf = df['Good_Investment']

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

In [35]:
mlflow.set_experiment("real_estate_classification")

clf_models = {
    "LogisticRegression": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]),
    "KNN": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier())]),
    "DecisionTree": Pipeline([("scaler", StandardScaler()), ("model", DecisionTreeClassifier(random_state=42))]),
    "RandomForest": Pipeline([("scaler", StandardScaler()), ("model", RandomForestClassifier(random_state=42, n_jobs=-1))]),
    "GradientBoosting": Pipeline([("scaler", StandardScaler()), ("model", GradientBoostingClassifier(random_state=42))]),
    "XGBoost": Pipeline([("scaler", StandardScaler()), ("model", XGBClassifier(random_state=42, n_jobs=-1))]),
    "LightGBM": Pipeline([("scaler", StandardScaler()), ("model", LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1))]),
}

In [36]:
def train_and_log_classifier(clf_models):
    for name, model in clf_models.items():
        with mlflow.start_run(run_name=name):
            model.fit(X_train_clf, y_train_clf)
            preds = model.predict(X_test_clf)
            proba = model.predict_proba(X_test_clf)[:, 1]

            metrics = classification_metrics(y_test_clf, preds, proba)

            mlflow.log_param("model_type", name)
            mlflow.log_metrics(metrics)

            signature = infer_signature(X_test_clf, preds)
            log_model_by_type(model, name, signature)

            print(f"{name}: Acc={metrics['accuracy']:.3f} Prec={metrics['precision']:.3f} "
                  f"Rec={metrics['recall']:.3f} F1={metrics['f1']:.3f} AUC={metrics['roc_auc']:.3f}")

In [37]:
train_and_log_classifier(clf_models=clf_models)

/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


LogisticRegression: Acc=0.714 Prec=0.446 Rec=0.162 F1=0.237 AUC=0.723
🏃 View run LogisticRegression at: http://127.0.0.1:5000/#/experiments/5/runs/dd3c5260776847f39cd7590609af6e63
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


KNN: Acc=0.684 Prec=0.365 Rec=0.197 F1=0.256 AUC=0.576
🏃 View run KNN at: http://127.0.0.1:5000/#/experiments/5/runs/cfb82a5fdf844ea18a8fd67d9b0a403b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


...

GradientBoosting: Acc=0.723 Prec=0.428 Rec=0.015 F1=0.029 AUC=0.722
🏃 View run GradientBoosting at: http://127.0.0.1:5000/#/experiments/5/runs/3ead7f889d294096a5ea84a7949177da
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


XGBoost: Acc=0.714 Prec=0.436 Rec=0.134 F1=0.205 AUC=0.723
🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/5/runs/84698af3311e4a208443923dd568473f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


LightGBM: Acc=0.723 Prec=0.428 Rec=0.011 F1=0.022 AUC=0.721
🏃 View run LightGBM at: http://127.0.0.1:5000/#/experiments/5/runs/7b10dcdffe06496fa570ffcc86ca6148
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


In [38]:
mlflow.set_experiment("real_estate_classification")
best_clf_models = {
    "LogisticRegression": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))]),
    "KNN": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier())]),
    "DecisionTree": Pipeline([("scaler", StandardScaler()), ("model", DecisionTreeClassifier(random_state=42, class_weight="balanced"))]),
    "RandomForest": Pipeline([("scaler", StandardScaler()), ("model", RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"))]),
    "GradientBoosting": Pipeline([("scaler", StandardScaler()), ("model", GradientBoostingClassifier(random_state=42))]),
    "XGBoost": Pipeline([("scaler", StandardScaler()), ("model", XGBClassifier(random_state=42, n_jobs=-1, scale_pos_weight=2.57))]),
    "LightGBM": Pipeline([("scaler", StandardScaler()), ("model", LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1, class_weight="balanced"))]),
}

In [39]:
train_and_log_classifier(clf_models=best_clf_models)

/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


LogisticRegression: Acc=0.681 Prec=0.450 Rec=0.713 F1=0.552 AUC=0.723
🏃 View run LogisticRegression at: http://127.0.0.1:5000/#/experiments/5/runs/57cf518057ce43a699ac5802fc8282f9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


KNN: Acc=0.684 Prec=0.365 Rec=0.197 F1=0.256 AUC=0.576
🏃 View run KNN at: http://127.0.0.1:5000/#/experiments/5/runs/dc0a9cb40c3f433180488f21ef4b6a22
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


...

GradientBoosting: Acc=0.723 Prec=0.428 Rec=0.015 F1=0.029 AUC=0.722
🏃 View run GradientBoosting at: http://127.0.0.1:5000/#/experiments/5/runs/000b80355f794ec7a48fc743f975901c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


XGBoost: Acc=0.670 Prec=0.446 Rec=0.819 F1=0.578 AUC=0.722
🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/5/runs/c9ae403929ab472c97c7b3ddd4dace58
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


LightGBM: Acc=0.669 Prec=0.446 Rec=0.831 F1=0.580 AUC=0.723
🏃 View run LightGBM at: http://127.0.0.1:5000/#/experiments/5/runs/e73327025ed04c07bb0a672e2f93c488
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


In [40]:
best_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1, class_weight="balanced")),
])
best_clf.fit(X_train_clf, y_train_clf)
proba = best_clf.predict_proba(X_test_clf)[:, 1]

In [41]:
from sklearn.metrics import precision_recall_curve

thresholds = np.arange(0.1, 0.9, 0.05)
results = []
for t in thresholds:
    preds_t = (proba >= t).astype(int)
    f1_t = f1_score(y_test_clf, preds_t)
    results.append((t, f1_t))
    print(f"threshold={t:.2f}  F1={f1_t:.3f}")

best_threshold = max(results, key=lambda x: x[1])[0]
print("\nBest threshold:", best_threshold)

threshold=0.10  F1=0.432
threshold=0.15  F1=0.433
threshold=0.20  F1=0.459
threshold=0.25  F1=0.557
threshold=0.30  F1=0.569
threshold=0.35  F1=0.575
threshold=0.40  F1=0.578
threshold=0.45  F1=0.580
threshold=0.50  F1=0.580
threshold=0.55  F1=0.580
threshold=0.60  F1=0.577
threshold=0.65  F1=0.564
threshold=0.70  F1=0.057
threshold=0.75  F1=0.003
threshold=0.80  F1=0.000
threshold=0.85  F1=0.000

Best threshold: 0.5000000000000001


In [42]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__num_leaves": [31, 50, 70, 100],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "model__max_depth": [-1, 5, 10, 20],
    "model__subsample": [0.7, 0.8, 1.0],
}

lgbm_clf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1, class_weight="balanced")),
])

search_clf = RandomizedSearchCV(
    estimator=lgbm_clf_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring="f1",    
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search_clf.fit(X_train_clf, y_train_clf)
print("Best params:", search_clf.best_params_)
print("Best CV F1:", round(search_clf.best_score_, 4))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best params: {'model__subsample': 0.7, 'model__num_leaves': 31, 'model__n_estimators': 100, 'model__max_depth': 20, 'model__learning_rate': 0.05}
Best CV F1: 0.5815


In [43]:
best_clf_tuned = search_clf.best_estimator_

preds = best_clf_tuned.predict(X_test_clf)
proba = best_clf_tuned.predict_proba(X_test_clf)[:, 1]

metrics = classification_metrics(y_test_clf, preds, proba)

print(f"Tuned LightGBM (test): Acc={metrics['accuracy']:.3f} Prec={metrics['precision']:.3f} "
      f"Rec={metrics['recall']:.3f} F1={metrics['f1']:.3f} AUC={metrics['roc_auc']:.3f}")

with mlflow.start_run(run_name="LightGBM_clf_tuned") as run:
    mlflow.log_params(search_clf.best_params_)
    mlflow.log_metrics(metrics)
    signature = infer_signature(X_test_clf, preds)
    log_model_by_type(best_clf_tuned, "LightGBM", signature)
    run_id_clf = run.info.run_id
print(run_id_clf)

Tuned LightGBM (test): Acc=0.669 Prec=0.446 Rec=0.833 F1=0.581 AUC=0.723


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run LightGBM_clf_tuned at: http://127.0.0.1:5000/#/experiments/5/runs/316bf90d71f94f6bbc5456e6abaa0cf6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
316bf90d71f94f6bbc5456e6abaa0cf6


In [44]:
mlflow.register_model(
    f"runs:/{run_id_clf}/model",
    "real_estate_classification_model"
)

Successfully registered model 'real_estate_classification_model'.
2026/08/27 14:41:56 WARNING mlflow.tracking._model_registry.fluent: Run with id 316bf90d71f94f6bbc5456e6abaa0cf6 has no artifacts at artifact path 'model', registering model based on models:/m-386b151ff26d44b5a6d5d7e0d9fd499a instead
2026/08/27 14:41:56 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: real_estate_classification_model, version 1
Created version '1' of model 'real_estate_classification_model'.


<ModelVersion: aliases=[], creation_timestamp=1787821916118, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1787821916118, metrics=None, model_id=None, name='real_estate_classification_model', params=None, run_id='316bf90d71f94f6bbc5456e6abaa0cf6', run_link='', source='models:/m-386b151ff26d44b5a6d5d7e0d9fd499a', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

### Classification Experiments

In [27]:
#df['size_per_floor'] = df['Size_in_SqFt'] / df['Total_Floors']
#df['school_hospital_sum'] = df['Nearby_Schools'] + df['Nearby_Hospitals']

In [28]:
#x_clf_cols = x_clf_cols + ['size_per_floor', 'school_hospital_sum']
#X_clf = df[x_clf_cols]

In [29]:
#mlflow.set_experiment("real_estate_classification")

In [30]:
#X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
#    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

In [31]:
#scaler_clf = StandardScaler()
#X_train_clf_scaled = pd.DataFrame(scaler_clf.fit_transform(X_train_clf),
#                                  columns=X_train_clf.columns, index=X_train_clf.index)
#X_test_clf_scaled = pd.DataFrame(scaler_clf.transform(X_test_clf),
#                                 columns=X_test_clf.columns, index=X_test_clf.index)

In [32]:
#train_and_log_classifier(best_clf_models)

In [33]:
#x_clf_cols = [c for c in x_clf_cols if c not in ['size_per_floor', 'school_hospital_sum']]
#len(x_clf_cols) 

### Exporting the Model

In [46]:
import mlflow
from mlflow.tracking import MlflowClient
model_name = ['real_estate_regression_model','real_estate_classification_model']
client = MlflowClient()
for name in model_name:
    mv = client.get_model_version(name, "1")
    run_id = mv.run_id
    print(run_id)

1cc65c2bbd294b6b8bd70f89cceb6ad2
316bf90d71f94f6bbc5456e6abaa0cf6


#### Download Models from MLflow

In [47]:
# downloads the model folder and returns where it landed
model_name = {'real_estate_regression_model':'artifacts/regression/skops','real_estate_classification_model':'artifacts/classification/skops'}
version = '1'
for name,path in model_name.items():
    local_path = mlflow.artifacts.download_artifacts(
        artifact_uri=f"models:/{name}/{version}",
        dst_path=model_name[name]
)
    print(local_path)

/Users/mukundanramesh/projects/Real Estate Investment Advisor - Predicting Property Profitability & Future Value/artifacts/regression/skops


/Users/mukundanramesh/projects/Real Estate Investment Advisor - Predicting Property Profitability & Future Value/artifacts/classification/skops


#### Save Models as pkl

In [48]:
import skops.io as sio
import pickle

skops_files = {"real_estate_regression_model":"artifacts/regression/skops/model.skops",
               "real_estate_classification_model":'artifacts/classification/skops/model.skops'}

for name,f in skops_files.items():
    untrusted = sio.get_untrusted_types(file=f)
    model = sio.load(f, trusted=untrusted)
    with open(f"artifacts/pkls/{name}.pkl", "wb") as out:
        pickle.dump(model, out)

In [49]:
import pickle

artifacts = {
    "x_reg_cols": x_reg_cols,    # <- from cell 9
    "x_clf_cols": x_clf_cols,    # <- from cell 20
}

for name, obj in artifacts.items():
    with open(f"artifacts/{name}.pkl", "wb") as f:
        pickle.dump(obj, f)

In [50]:
def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)

reg_model  = load_pkl("artifacts/pkls/real_estate_regression_model.pkl")
clf_model  = load_pkl("artifacts/pkls/real_estate_classification_model.pkl")
x_reg_cols = load_pkl("artifacts/x_reg_cols.pkl")
x_clf_cols = load_pkl("artifacts/x_clf_cols.pkl")

### Reducing Number of inputs for models

In [52]:
x_reg_cols = ['BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Nearby_Schools',
    'Nearby_Hospitals', 'Public_Transport_Accessibility', 'Parking_Space',
    'Security', 'Amenities_Count',
    'City_Ahmedabad', 'City_Amritsar', 'City_Bangalore', 'City_Bhopal',
    'City_Bhubaneswar', 'City_Bilaspur', 'City_Chennai', 'City_Coimbatore',
    'City_Cuttack', 'City_Dehradun', 'City_Durgapur', 'City_Dwarka',
    'City_Faridabad', 'City_Gaya', 'City_Gurgaon', 'City_Guwahati',
    'City_Haridwar', 'City_Hyderabad', 'City_Indore', 'City_Jaipur',
    'City_Jamshedpur', 'City_Jodhpur', 'City_Kochi', 'City_Kolkata',
    'City_Lucknow', 'City_Ludhiana', 'City_Mangalore', 'City_Mumbai',
    'City_Mysore', 'City_Nagpur', 'City_New_Delhi', 'City_Noida',
    'City_Patna', 'City_Pune', 'City_Raipur', 'City_Ranchi', 'City_Silchar',
    'City_Surat', 'City_Trivandrum', 'City_Vijayawada',
    'City_Vishakhapatnam', 'City_Warangal', 'Property_Type_Apartment',
    'Property_Type_Independent_House', 'Property_Type_Villa']
X_reg = df[x_reg_cols]
y_reg = df['Future_Price_5Y']
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

In [53]:
train_and_log_regressor(models=models)

/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


LinearRegression: RMSE=45.33  MAE=33.69  R2=0.946
🏃 View run LinearRegression at: http://127.0.0.1:5000/#/experiments/5/runs/7431bf49c80f4591944ba1263f83f9b3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Ridge: RMSE=45.33  MAE=33.69  R2=0.946
🏃 View run Ridge at: http://127.0.0.1:5000/#/experiments/5/runs/494a6618ec0a4eba913b6096fa2b9918
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


...

GradientBoosting: RMSE=45.35  MAE=32.01  R2=0.946
🏃 View run GradientBoosting at: http://127.0.0.1:5000/#/experiments/5/runs/6037aced2529442ba3f77f9635716776
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


XGBoost: RMSE=42.94  MAE=30.17  R2=0.952
🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/5/runs/e321ec21661246549658a605c59c906a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


/Users/mukundanramesh/projects/project_venvs/guvi-python-sql-pandas-project-3/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


LightGBM: RMSE=42.54  MAE=29.94  R2=0.953
🏃 View run LightGBM at: http://127.0.0.1:5000/#/experiments/5/runs/ebc75c0b66224b3abc13bdc47428a1e0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
